# Assignment 1: The Headline Test
**Student:** Bharat | **ERP ID:** 37303

This notebook checks the supplied teaching dataset and explores price movements. It does not treat the illustrative records as official market statistics.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA_PATH = Path("teaching_prices_clean.csv")
df = pd.read_csv(DATA_PATH)

df.head()

## 1. Dataset structure and data dictionary

In [ ]:
print("Actual shape (rows, columns):", df.shape)
print("Distinct items:", df["item"].nunique())
print("Distinct dates:", df["date"].nunique())
print("Distinct cities:", df["city"].nunique())
print("\\nColumn names:", df.columns.tolist())
print("\\nData types:\\n", df.dtypes)
print("\\nMissing values:\\n", df.isna().sum())

### What the columns mean
- `date`: date of the recorded observation
- `item`: anonymized product/item label
- `city`: anonymized city label
- `price`: numeric recorded price
- `unit`: unit attached to the price
- `data_label`: source/status note

The file's `data_label` identifies these records as illustrative teaching data, not actual PBS observations.

## 2. Clean checks and coverage

In [ ]:
# Convert date to datetime and inspect the observation period
df["date"] = pd.to_datetime(df["date"], errors="coerce")
print("Date range:", df["date"].min().date(), "to", df["date"].max().date())
print("Duplicate rows:", df.duplicated().sum())
print("Unique units:", df["unit"].unique())
print("Data labels:", df["data_label"].unique())

# Check whether each item-city combination has one observation per date
coverage = df.groupby(["item", "city"])["date"].nunique().unstack()
coverage

## 3. Question 1 — How did the average recorded price change over the observed dates?

In [ ]:
# Average across all recorded item-city observations for each date
daily_mean = df.groupby("date", as_index=False)["price"].mean()
display(daily_mean)

first_mean = daily_mean.iloc[0]["price"]
last_mean = daily_mean.iloc[-1]["price"]
absolute_change = last_mean - first_mean
percent_change = (absolute_change / first_mean) * 100

print(f"First-date mean: {first_mean:.3f}")
print(f"Last-date mean:  {last_mean:.3f}")
print(f"Absolute change: {absolute_change:.3f}")
print(f"Percentage change: {percent_change:.2f}%")

plt.figure(figsize=(9, 5))
plt.plot(daily_mean["date"], daily_mean["price"], marker="o")
plt.title("Mean recorded price across all item-city observations")
plt.xlabel("Observation date")
plt.ylabel("Mean price (dataset unit)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

**Interpretation prompt:** Describe the direction and size of the change using the calculated values. Remember that this is a simple average across different items, so it is not a formal consumer price index and does not account for household spending weights.

## 4. Question 2 — Which item has the greatest variation in recorded price?

In [ ]:
item_stats = (
    df.groupby("item")["price"]
      .agg(count="count", mean="mean", std="std", minimum="min", maximum="max")
      .sort_values("std", ascending=False)
)
display(item_stats)

most_variable_item = item_stats.index[0]
print("Largest standard deviation:", most_variable_item)
print("Standard deviation:", item_stats.iloc[0]["std"])

Standard deviation measures spread around an item's mean price. Comparing standard deviations is most meaningful here because the items share the same recorded unit, but the result still describes only the observations in this teaching dataset.

## 5. Compare average price by item and city

In [ ]:
item_city_means = df.pivot_table(
    index="item", columns="city", values="price", aggfunc="mean"
)
display(item_city_means)

item_date_mean = df.groupby(["date", "item"], as_index=False)["price"].mean()
plt.figure(figsize=(10, 6))
for item_name, group in item_date_mean.groupby("item"):
    plt.plot(group["date"], group["price"], marker="o", label=item_name)
plt.title("Average recorded price by item over time")
plt.xlabel("Observation date")
plt.ylabel("Mean price across cities")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 6. Limitations and conclusion

- The dataset contains anonymized item and city labels, and its records are explicitly illustrative.
- It covers a limited set of four items, two cities, and twelve dates; it cannot represent all prices or all households.
- The data can describe recorded price patterns, but cannot explain causes such as taxes, supply disruptions, weather, or policy changes.
- A simple average across unlike products is not an official inflation measure.

